# Stage 1: Canonical Detection Layer\n\nThis notebook trains and evaluates **YOLO26m** and **YOLO11x** for canonical schematic component detection.\n\nOutputs per image:\n- component boxes\n- component class\n- confidence\n- orientation is deferred to Stage 1B as a separate crop classifier\n\nMethod choices grounded in the papers you collected:\n- `AMSNet`: strong component detection is feasible, but topology is harder than detection\n- `Netlistify` / `Image2Net`: keep detection and orientation separate\n- `SINA`: strong detector + downstream symbolic logic is the right decomposition\n- `OmniSch`: do not rely on generic multimodal reasoning for structured schematic parsing\n\nAccuracy-first training policy:\n1. train on the full merged canonical dataset\n2. fine-tune on a **source-balanced** training manifest to recover cross-source recall\n3. compare `YOLO26m` vs `YOLO11x` using source-aware validation\n

In [ ]:
!pip install -U ultralytics pyyaml pandas seaborn scikit-learn pillow

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
from pathlib import Path\nimport ast\nimport json\nimport os\nimport random\nimport re\nfrom collections import Counter, defaultdict\n\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom PIL import Image, ImageOps\n\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\n\n# Update these paths for your Drive layout.\nDATA_ROOT = Path('/content/drive/MyDrive/Datasets')\nDATASET_ROOT = DATA_ROOT / 'ALL_COMPONENTS.merged.yolov8'\nWORK_ROOT = DATA_ROOT / 'stage1_detection_workspace'\nWORK_ROOT.mkdir(parents=True, exist_ok=True)\n\nIMGSZ_CANDIDATES = [960, 1280]\nMODELS = ['yolo26m.pt', 'yolo11x.pt']\nEPOCHS_FULL = 80\nEPOCHS_BALANCED_FINETUNE = 30\nBATCH = 16\nDEVICE = 0\nPROJECT_NAME = 'stage1_canonical_detection'\n\n# Accuracy-first balance rule: cap the dominant source for fine-tune.\nBALANCE_CAP_PER_SOURCE = 12000\n\n# Cleaning policy\nAPPLY_EXIF_TRANSPOSE = True\nFORCE_GRAYSCALE_EXPORT = False  # keep False by default; use only as ablation\nMIN_BOX_WH = 1e-4  # normalized YOLO coords\n\nassert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'\nprint('DATASET_ROOT =', DATASET_ROOT)\nprint('WORK_ROOT =', WORK_ROOT)

In [ ]:
def load_names(dataset_root: Path):\n    text = (dataset_root / 'data.yaml').read_text(encoding='utf-8')\n    return ast.literal_eval(re.search(r'names:\\s*(\\[.*\\])', text, re.S).group(1))\n\n\ndef image_paths(split: str):\n    return sorted((DATASET_ROOT / split / 'images').glob('*'))\n\n\ndef label_path_for_image(p: Path):\n    return p.parent.parent / 'labels' / f'{p.stem}.txt'\n\n\ndef detect_source_from_name(name: str):\n    name = name.lower()\n    if 'schematic_images_hf' in name:\n        return 'schematic_images_hf'\n    if name.startswith('cghd'):\n        return 'cghd'\n    if name.startswith('ci2n'):\n        return 'ci2n'\n    if name.startswith('digitize_hcd'):\n        return 'digitize_hcd'\n    if 'schematic_merged' in name or 'schematic.merged' in name:\n        return 'schematic_merged'\n    if name.startswith('v3qwe'):\n        return 'v3qwe'\n    return 'other'\n\n\ndef validate_yolo_label_line(line: str, nc: int):\n    parts = line.strip().split()\n    if len(parts) < 5:\n        return None\n    try:\n        cls = int(float(parts[0]))\n        x, y, w, h = [float(v) for v in parts[1:5]]\n    except Exception:\n        return None\n    if not (0 <= cls < nc):\n        return None\n    x = min(max(x, 0.0), 1.0)\n    y = min(max(y, 0.0), 1.0)\n    w = min(max(w, 0.0), 1.0)\n    h = min(max(h, 0.0), 1.0)\n    if w < MIN_BOX_WH or h < MIN_BOX_WH:\n        return None\n    return f'{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}'\n\n\ndef audit_dataset(dataset_root: Path):\n    names = load_names(dataset_root)\n    nc = len(names)\n    report = []\n    class_counts = Counter()\n    bad_files = []\n    source_counts = defaultdict(Counter)\n\n    for split in ['train', 'valid', 'test']:\n        images = image_paths(split)\n        for image_path in images:\n            src = detect_source_from_name(image_path.name)\n            source_counts[split][src] += 1\n            label_path = label_path_for_image(image_path)\n            if not label_path.exists():\n                bad_files.append((split, str(image_path), 'missing_label'))\n                continue\n            for raw in label_path.read_text(encoding='utf-8').splitlines():\n                fixed = validate_yolo_label_line(raw, nc)\n                if fixed is None:\n                    bad_files.append((split, str(label_path), raw))\n                    continue\n                cls = int(fixed.split()[0])\n                class_counts[cls] += 1\n\n        report.append({'split': split, 'images': len(images), **dict(source_counts[split])})\n\n    class_df = pd.DataFrame([\n        {'class_id': i, 'class_name': names[i], 'instances': class_counts[i]}\n        for i in range(nc)\n    ]).sort_values(['instances', 'class_name'], ascending=[False, True])\n    return pd.DataFrame(report), class_df, pd.DataFrame(bad_files, columns=['split', 'path', 'issue'])\n\n\nsplit_df, class_df, bad_df = audit_dataset(DATASET_ROOT)\ndisplay(split_df)\ndisplay(class_df.head(20))\ndisplay(class_df.tail(20))\nprint('Bad label/file entries:', len(bad_df))

## Cleaning Policy\n\nUse these rules for maximum accuracy:\n- **Do not** force a global auto-rotation heuristic. In schematics, rotation is often semantic.\n- Apply **EXIF transpose only** if image metadata demands it.\n- **Do not** force grayscale by default. Use grayscale export only as an ablation. Some detectors benefit from RGB pretraining even on line drawings.\n- Clean labels first: remove malformed rows, clamp boxes, drop degenerate tiny boxes.\n- If `bad_df` is empty or tiny, train on the original dataset and avoid a huge image copy.\n

In [ ]:
def write_list_file(paths, out_path: Path):\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    out_path.write_text('\\n'.join(str(p.resolve()) for p in paths) + '\\n', encoding='utf-8')\n    return out_path\n\n\ndef make_source_manifests():\n    manifests = {}\n    names = load_names(DATASET_ROOT)\n    for split in ['train', 'valid', 'test']:\n        imgs = image_paths(split)\n        by_source = defaultdict(list)\n        for p in imgs:\n            by_source[detect_source_from_name(p.name)].append(p)\n\n        manifests[split] = {}\n        for source, paths in by_source.items():\n            txt_path = write_list_file(paths, WORK_ROOT / 'lists' / f'{split}_{source}.txt')\n            yaml_path = WORK_ROOT / 'yamls' / f'{split}_{source}.yaml'\n            yaml_path.parent.mkdir(parents=True, exist_ok=True)\n            yaml.safe_dump({\n                'path': str(DATASET_ROOT.resolve()),\n                'train': str(txt_path) if split == 'train' else str(WORK_ROOT / 'lists' / 'train_full.txt'),\n                'val': str(txt_path) if split == 'valid' else str(WORK_ROOT / 'lists' / 'valid_full.txt'),\n                'test': str(txt_path) if split == 'test' else str(WORK_ROOT / 'lists' / 'test_full.txt'),\n                'nc': len(names),\n                'names': names,\n            }, yaml_path.open('w', encoding='utf-8'), sort_keys=False)\n            manifests[split][source] = yaml_path\n    return manifests\n\n\ndef build_full_and_balanced_manifests(balance_cap=BALANCE_CAP_PER_SOURCE):\n    names = load_names(DATASET_ROOT)\n    train_imgs = image_paths('train')\n    valid_imgs = image_paths('valid')\n    test_imgs = image_paths('test')\n\n    write_list_file(train_imgs, WORK_ROOT / 'lists' / 'train_full.txt')\n    write_list_file(valid_imgs, WORK_ROOT / 'lists' / 'valid_full.txt')\n    write_list_file(test_imgs, WORK_ROOT / 'lists' / 'test_full.txt')\n\n    by_source = defaultdict(list)\n    for p in train_imgs:\n        by_source[detect_source_from_name(p.name)].append(p)\n\n    balanced = []\n    rng = random.Random(SEED)\n    for source, paths in sorted(by_source.items()):\n        if len(paths) > balance_cap:\n            balanced.extend(rng.sample(paths, balance_cap))\n        else:\n            balanced.extend(paths)\n    balanced = sorted(balanced)\n    write_list_file(balanced, WORK_ROOT / 'lists' / 'train_balanced.txt')\n\n    full_yaml = WORK_ROOT / 'yamls' / 'full.yaml'\n    balanced_yaml = WORK_ROOT / 'yamls' / 'balanced.yaml'\n    full_yaml.parent.mkdir(parents=True, exist_ok=True)\n\n    full_cfg = {\n        'path': str(DATASET_ROOT.resolve()),\n        'train': str((WORK_ROOT / 'lists' / 'train_full.txt').resolve()),\n        'val': str((WORK_ROOT / 'lists' / 'valid_full.txt').resolve()),\n        'test': str((WORK_ROOT / 'lists' / 'test_full.txt').resolve()),\n        'nc': len(names),\n        'names': names,\n    }\n    balanced_cfg = {\n        'path': str(DATASET_ROOT.resolve()),\n        'train': str((WORK_ROOT / 'lists' / 'train_balanced.txt').resolve()),\n        'val': str((WORK_ROOT / 'lists' / 'valid_full.txt').resolve()),\n        'test': str((WORK_ROOT / 'lists' / 'test_full.txt').resolve()),\n        'nc': len(names),\n        'names': names,\n    }\n    yaml.safe_dump(full_cfg, full_yaml.open('w', encoding='utf-8'), sort_keys=False)\n    yaml.safe_dump(balanced_cfg, balanced_yaml.open('w', encoding='utf-8'), sort_keys=False)\n    return full_yaml, balanced_yaml, pd.Series({k: len(v) for k, v in by_source.items()}).sort_values(ascending=False)\n\n\nsource_manifests = make_source_manifests()\nFULL_YAML, BALANCED_YAML, train_source_counts = build_full_and_balanced_manifests()\ndisplay(train_source_counts.rename('train_images'))\nprint('FULL_YAML =', FULL_YAML)\nprint('BALANCED_YAML =', BALANCED_YAML)

## Training Plan\n\nAccuracy-first recipe:\n1. train each detector on the **full** merged data\n2. fine-tune the same run on the **balanced** source-capped training manifest\n3. compare on full validation + source-specific validation\n\nWhy this matters: your merged train split is currently dominated by `schematic_images.hf`, so raw training alone will bias the detector toward that style family.\n

In [ ]:
from ultralytics import YOLO\n\ndef train_detector(model_name, imgsz):\n    run_slug = f"{Path(model_name).stem}_img{imgsz}"\n    model = YOLO(model_name)\n    model.train(\n        data=str(FULL_YAML),\n        imgsz=imgsz,\n        epochs=EPOCHS_FULL,\n        batch=BATCH,\n        seed=SEED,\n        device=DEVICE,\n        project=str(WORK_ROOT / PROJECT_NAME),\n        name=run_slug + '_full',\n        pretrained=True,\n        optimizer='auto',\n        cos_lr=True,\n        close_mosaic=10,\n        patience=25,\n        degrees=180,\n        translate=0.02,\n        scale=0.25,\n        shear=0.0,\n        perspective=0.0,\n        fliplr=0.0,\n        flipud=0.0,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        mosaic=0.6,\n        mixup=0.0,\n        copy_paste=0.0,\n        erasing=0.0,\n        plots=True\n    )\n\n    full_best = WORK_ROOT / PROJECT_NAME / f'{run_slug}_full' / 'weights' / 'best.pt'\n    finetune_model = YOLO(str(full_best))\n    finetune_model.train(\n        data=str(BALANCED_YAML),\n        imgsz=imgsz,\n        epochs=EPOCHS_BALANCED_FINETUNE,\n        batch=BATCH,\n        seed=SEED,\n        device=DEVICE,\n        project=str(WORK_ROOT / PROJECT_NAME),\n        name=run_slug + '_balanced_ft',\n        pretrained=False,\n        cos_lr=True,\n        close_mosaic=0,\n        patience=15,\n        degrees=180,\n        translate=0.02,\n        scale=0.15,\n        shear=0.0,\n        perspective=0.0,\n        fliplr=0.0,\n        flipud=0.0,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        mosaic=0.0,\n        mixup=0.0,\n        copy_paste=0.0,\n        erasing=0.0,\n        lr0=1e-3,\n        plots=True\n    )\n\n\n# Uncomment to run.\n# for model_name in MODELS:\n#     for imgsz in IMGSZ_CANDIDATES:\n#         train_detector(model_name, imgsz)

In [ ]:
from ultralytics import YOLO\n\ndef collect_metrics_obj(m):\n    d = getattr(m, 'results_dict', {}) or {}\n    row = {\n        'precision': float(d.get('metrics/precision(B)', np.nan)),\n        'recall': float(d.get('metrics/recall(B)', np.nan)),\n        'mAP50': float(d.get('metrics/mAP50(B)', np.nan)),\n        'mAP50_95': float(d.get('metrics/mAP50-95(B)', np.nan)),\n    }\n    if hasattr(m, 'box') and hasattr(m.box, 'maps'):\n        row['per_class_map50_95'] = list(map(float, m.box.maps))\n    return row\n\n\ndef evaluate_model(weights_path, split='valid'):\n    model = YOLO(str(weights_path))\n    rows = []\n\n    full_yaml = FULL_YAML if split == 'valid' else FULL_YAML\n    full_res = model.val(data=str(full_yaml), split='val' if split == 'valid' else 'test', imgsz=1280, batch=BATCH, device=DEVICE, plots=False, augment=True)\n    full_row = collect_metrics_obj(full_res)\n    full_row.update({'source': 'ALL', 'split': split})\n    rows.append(full_row)\n\n    for source, yaml_path in source_manifests['valid' if split == 'valid' else 'test'].items():\n        res = model.val(data=str(yaml_path), split='val' if split == 'valid' else 'test', imgsz=1280, batch=BATCH, device=DEVICE, plots=False, augment=True)\n        row = collect_metrics_obj(res)\n        row.update({'source': source, 'split': split})\n        rows.append(row)\n\n    return pd.DataFrame(rows)\n\n\n# Example usage after training:\n# best_weights = WORK_ROOT / PROJECT_NAME / 'yolo26m_img1280_balanced_ft' / 'weights' / 'best.pt'\n# eval_df = evaluate_model(best_weights, split='valid')\n# display(eval_df.sort_values(['source']))

In [ ]:
def rank_models(eval_tables):\n    rows = []\n    for model_tag, df in eval_tables.items():\n        core = df[df['source'] != 'ALL'].copy()\n        rows.append({\n            'model_tag': model_tag,\n            'mean_source_recall': core['recall'].mean(),\n            'min_source_recall': core['recall'].min(),\n            'mean_source_map50_95': core['mAP50_95'].mean(),\n            'overall_map50_95': float(df[df['source'] == 'ALL']['mAP50_95'].iloc[0]),\n            'overall_recall': float(df[df['source'] == 'ALL']['recall'].iloc[0]),\n        })\n    return pd.DataFrame(rows).sort_values(\n        ['mean_source_recall', 'min_source_recall', 'mean_source_map50_95', 'overall_map50_95'],\n        ascending=False\n    )\n\n\n# Example:\n# eval_tables = {\n#     'yolo26m_img1280': evaluate_model(path1, split='valid'),\n#     'yolo11x_img1280': evaluate_model(path2, split='valid'),\n# }\n# display(rank_models(eval_tables))

## Decision Rule\n\nPick the detector by this order:\n1. mean source-wise recall\n2. minimum source recall\n3. mean source-wise mAP50-95\n4. overall full-validation mAP50-95\n\nAfter freezing the detector, Stage 1B is the orientation classifier on GT and predicted crops for orientation-sensitive classes only.\n\nDefault advice on cleaning for these datasets:\n- clean labels, yes\n- EXIF transpose only if needed\n- no forced grayscale unless you run it as an ablation\n- no heuristic global rotation normalization\n